# Prepare CLIP data
This notebook extracts from paintings the crops of objects needed to fine-tune CLIP and stores the descriptions together with image object paths in a csv.

### 0. Import libraries and load data

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import copy
import shutil

sys.path.append("../annotate_dataset/")

import tiktoken
import numpy as np
import polars as pl
from tqdm import tqdm
from PIL import ImageOps
from ground_objects import *
from annotate_paintings_utils import *

SET_NAME = "train"
INPUT_PATH_EMBEDDINGS = "../../data/embeddings/"
EMBEDDINGS_FILE_NAME = f"baseline_embeddings_{SET_NAME}_trained.json"
STORAGE_PATH = "../../data/fine-tuning_clip/"
VERBOSE = False
TARGET_SIZE = 224
PADDING_COLOR = 127

In [ ]:
embeddings_data = pl.read_json(f"{INPUT_PATH_EMBEDDINGS}{EMBEDDINGS_FILE_NAME}").explode(pl.all())

if SET_NAME != "test":
    embeddings_data = (
        embeddings_data.sort("probability", descending=True)
        .group_by("text", maintain_order=True)
        .agg(pl.all().first())
        .sort("painting_id")
    )

embeddings_data = embeddings_data.with_row_index()
embeddings_data

### 1. Get image object crops and store them

In [ ]:
def get_box_coordinates(box, width, height):
    updated_box = box * np.array([width, height, width, height])

    updated_box[:2] -= updated_box[2:] / 2
    updated_box[2:] += updated_box[:2]

    return list(updated_box)

In [ ]:
try:
    shutil.rmtree(f"{STORAGE_PATH}{SET_NAME}")
except:
    pass

os.mkdir(f"{STORAGE_PATH}{SET_NAME}")

In [ ]:
for bbox_index in tqdm(range(embeddings_data.shape[0])):
    # get image, bounding box, probability and description
    painting_id = embeddings_data["painting_id"].to_list()[bbox_index]
    _, image = load_image(painting_id)
    width, height = image.size

    bbox = get_box_coordinates(embeddings_data["bounding_box"].to_list()[bbox_index], width, height)
    probability = embeddings_data["probability"].to_list()[bbox_index]
    description = embeddings_data["text"].to_list()[bbox_index]

    if VERBOSE:
        display_annotated_image(copy.deepcopy(image), [[description, probability, list(bbox)]])

    # crop the image object
    image_object = image.crop(bbox)

    # resize the image object
    image_object_width, image_object_height = image_object.size
    scale = min(TARGET_SIZE / image_object_width, TARGET_SIZE / image_object_height)
    new_width, new_height = int(image_object_width * scale), int(image_object_height * scale)
    image_object_resized = image_object.resize((new_width, new_height), Image.LANCZOS)

    # add padding
    image_object_resized_w_padding = ImageOps.expand(
        image_object_resized,
        border=(
            (TARGET_SIZE - new_width) // 2,
            (TARGET_SIZE - new_height) // 2,
            (TARGET_SIZE - new_width) - (TARGET_SIZE - new_width) // 2,
            (TARGET_SIZE - new_height) - (TARGET_SIZE - new_height) // 2 
        ),
        fill=(PADDING_COLOR, PADDING_COLOR, PADDING_COLOR)
    )

    # store image
    image_object_resized_w_padding.save(f"{STORAGE_PATH}{SET_NAME}/{bbox_index}.png")

### 2. Store the CSV file needed for training, validation and testing

In [ ]:
description_image_name = embeddings_data.select("index", "text").rename({"text": "description"}).with_columns((f"{SET_NAME}/" + pl.col("index").cast(pl.String) + ".png").alias("image_name"))
description_image_name

In [ ]:
description_image_name.write_csv(f"{STORAGE_PATH}{SET_NAME}.csv")

### 3. Count how many descriptions are longer than the limit of 77 tokens

In [ ]:
set_name = "train"
encoding = tiktoken.encoding_for_model("gpt-2")

data = pl.read_csv(f"{STORAGE_PATH}{set_name}.csv").with_columns(pl.col("description").map_elements(lambda x: len(encoding.encode(x)), return_dtype=pl.Int32).alias("tokens_no")).sort("tokens_no")
percentage_long_descriptions = data.filter(pl.col("tokens_no") > 77).shape[0] / data.shape[0] * 100
print(f"{round(percentage_long_descriptions, 2)}% of descriptions are longer than 77 tokens.")